# Structured HK diagnostic

Targeted experiment to isolate why abundant-data learning underperforms single-shot on the structured Hegselmann-Krause environments.

In [1]:

from __future__ import annotations

import contextlib
import hashlib
import inspect
import io
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 300)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing opinion_dynamics/.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.baseline import centrality_based_continuous_control
from opinion_dynamics.identify_nonlinear_unbounded import (
    GraphIdentifierEnv,
    pairs_from_intermediate,
)
import opinion_dynamics.experiments.online_single_shot_unbounded as online_single_shot

try:
    GIT_COMMIT = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repository root:", REPO_ROOT)
print("Git commit:", GIT_COMMIT)
print("identifier:", inspect.getsourcefile(GraphIdentifierEnv))
print("online_single_shot:", inspect.getsourcefile(online_single_shot))
print("NetworkGraph:", inspect.getsourcefile(NetworkGraph))


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_102212\4201529446.py:18: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: d:\Work\repos\RL\unknown_graph_networks
Git commit: 0eca8fe
identifier: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\identify_nonlinear_unbounded.py
online_single_shot: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\online_single_shot_unbounded.py
NetworkGraph: c:\Users\Chainsword\anaconda3\envs\phd_rl_algos\Lib\site-packages\rl_envs_forge\envs\network_graph\network_graph.py


In [2]:

# Parallel worker configuration.
NUM_SHARDS = 8
SHARD_ENV = "HK_DIAG_SHARD_ID"
THREAD_ENV = "HK_DIAG_TORCH_THREADS"

_shard_value = os.environ.get(SHARD_ENV)
WORKER_MODE = _shard_value is not None
SHARD_ID = int(_shard_value) if WORKER_MODE else None
if WORKER_MODE and SHARD_ID not in range(NUM_SHARDS):
    raise ValueError(f"{SHARD_ENV} must be in 0..{NUM_SHARDS - 1}; got {SHARD_ID}")

TORCH_THREADS = int(os.environ.get(THREAD_ENV, "2"))
torch.set_num_threads(max(1, TORCH_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = os.environ.get("HK_DIAG_DEVICE", "cpu").lower()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable.")

STUDY_NAME = "structured_hk_diagnostic_2026_10_02"
PIPELINE_VERSION = "2026-10-02-structured-hk-diagnostic-v1"

N = 15
TARGET = 1.0
NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1
MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)
LAMBDA_MIX = 1.0

FIT_LR = 1e-3
FIT_STEPS_STANDARD = 1_000
FIT_STEPS_LONG = 20_000
FIT_MAE_STOP = 5e-4
FIT_BATCH_SIZE = 256
FIT_CHECK_EVERY = 200
IDENTIFIER_KWARGS = {"hidden_dim": 16}
LEARNING_SEEDS = [0, 1, 2]

ABUNDANT_RESETS = 25
ABUNDANT_CAMPAIGNS_PER_RESET = 20
DATA_SEED_BASE = 1_200_000

STRUCTURED_HK_EPSILON = 0.25
STRUCTURED_ENVIRONMENT_SEEDS = [0, 1, 2, 3, 4]
EDGE_WEIGHT_JITTER_FRAC = 0.05
HK_OPINION_JITTER = 0.006

HK_FAMILIES = {
    "ring_bridge": {
        "peer_pattern": "ring",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.40,
        "bridge_weight": 3.0,
    },
    "paired_bridge": {
        "peer_pattern": "paired",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.55,
        "bridge_weight": 3.0,
    },
    "dense_weak_bridge": {
        "peer_pattern": "dense",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.08,
        "bridge_weight": 3.0,
    },
}

HK_LOW_CLUSTER = tuple(range(0, 7))
HK_HIGH_CLUSTER = tuple(range(7, 15))
HK_LOW_HUB = 0
HK_HIGH_HUB = 7

RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"
RESULTS_DIR = RESULTS_ROOT / "experiment_2026_10_02_structured_hk_diagnostic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = RESULTS_ROOT / "_trial_cache" / STUDY_NAME
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Old completed runs are reused whenever available.
SOURCE_ABUNDANT_STUDIES = [
    "final_unregularized_abundant_matched_distribution_25x20_2026_10_02",
    "abundant_matched_distribution_25x20_none_vs_gini_2026_10_02",
]
SOURCE_SINGLE_SHOT_STUDY = "final_unregularized_single_shot_2026_09_30"

VARIANT_ORDER = [
    "abundant_random_1k",
    "abundant_random_20k",
    "abundant_trueA_data_1k",
    "abundant_ss_replay_data_1k",
    "ss_exact_offline_1k",
    "ss_exact_offline_20k",
    "single_shot_online",
]

print("Mode:", "worker" if WORKER_MODE else "analysis/check")
if WORKER_MODE:
    print("Shard:", f"{SHARD_ID + 1}/{NUM_SHARDS}")
print("Device:", DEVICE)
print("Threads:", torch.get_num_threads())
print("Campaign budget:", B_CAMPAIGN)
print("Variants:", VARIANT_ORDER)


Mode: analysis/check
Device: cpu
Threads: 2
Campaign budget: 0.3157894736842105
Variants: ['abundant_random_1k', 'abundant_random_20k', 'abundant_trueA_data_1k', 'abundant_ss_replay_data_1k', 'ss_exact_offline_1k', 'ss_exact_offline_20k', 'single_shot_online']


In [3]:

def _normalize_rows(raw: np.ndarray) -> np.ndarray:
    raw = np.asarray(raw, dtype=float).copy()
    np.fill_diagonal(raw, 0.0)
    sums = raw.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise RuntimeError("Every graph row must have positive mass.")
    A = raw / sums
    np.fill_diagonal(A, 0.0)
    return A


def _jitter_positive_edges(
    raw: np.ndarray,
    rng: np.random.Generator,
    frac: float,
) -> np.ndarray:
    out = np.asarray(raw, dtype=float).copy()
    mask = out > 0
    multipliers = rng.uniform(
        1.0 - float(frac),
        1.0 + float(frac),
        size=out.shape,
    )
    out[mask] *= multipliers[mask]
    np.fill_diagonal(out, 0.0)
    return out


def _add_undirected(raw: np.ndarray, i: int, j: int, weight: float) -> None:
    raw[i, j] += float(weight)
    raw[j, i] += float(weight)


def centrality_from_A(A: np.ndarray) -> np.ndarray:
    v = compute_eigenvector_centrality(
        compute_laplacian(np.asarray(A, dtype=float))
    )
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    if v.sum() < 0:
        v = -v
    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)
    if v.sum() <= 1e-12:
        return np.full(N, 1.0 / N)
    return v / v.sum()


def build_hk_graph(
    family_name: str,
    environment_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    params = HK_FAMILIES[family_name]
    raw = np.zeros((N, N), dtype=float)

    for node in HK_LOW_CLUSTER:
        if node != HK_LOW_HUB:
            _add_undirected(
                raw,
                HK_LOW_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for node in HK_HIGH_CLUSTER:
        if node != HK_HIGH_HUB:
            _add_undirected(
                raw,
                HK_HIGH_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for cluster, hub in [
        (HK_LOW_CLUSTER, HK_LOW_HUB),
        (HK_HIGH_CLUSTER, HK_HIGH_HUB),
    ]:
        leaves = [node for node in cluster if node != hub]
        pattern = params["peer_pattern"]
        w = float(params["peer_weight"])

        if pattern == "ring":
            for left, right in zip(leaves, leaves[1:] + leaves[:1]):
                _add_undirected(raw, left, right, w)
        elif pattern == "paired":
            for index in range(0, len(leaves) - 1, 2):
                _add_undirected(
                    raw,
                    leaves[index],
                    leaves[index + 1],
                    w,
                )
            if len(leaves) % 2 == 1:
                _add_undirected(
                    raw,
                    leaves[-1],
                    leaves[0],
                    0.5 * w,
                )
        elif pattern == "dense":
            for pos, left in enumerate(leaves):
                for right in leaves[pos + 1:]:
                    _add_undirected(raw, left, right, w)
        else:
            raise ValueError(pattern)

    _add_undirected(
        raw,
        HK_LOW_HUB,
        HK_HIGH_HUB,
        params["bridge_weight"],
    )

    rng = np.random.default_rng(
        10_000
        + 100 * list(HK_FAMILIES).index(family_name)
        + int(environment_seed)
    )
    raw = _jitter_positive_edges(
        raw,
        rng,
        EDGE_WEIGHT_JITTER_FRAC,
    )
    return raw, _normalize_rows(raw)


def build_hk_x0(environment_seed: int) -> np.ndarray:
    base = np.array(
        [
            0.35, 0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
            0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71,
        ],
        dtype=float,
    )
    rng = np.random.default_rng(20_000 + int(environment_seed))
    return np.clip(
        base
        + rng.uniform(
            -HK_OPINION_JITTER,
            HK_OPINION_JITTER,
            size=N,
        ),
        0.0,
        1.0,
    )


ENVIRONMENT_SPECS: list[dict[str, Any]] = []
for family_index, family in enumerate(HK_FAMILIES):
    for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
        raw, A = build_hk_graph(family, environment_seed)
        x0 = build_hk_x0(environment_seed)
        # In the broad benchmark the 150 unstructured environments precede HK.
        source_environment_index = (
            150
            + family_index * len(STRUCTURED_ENVIRONMENT_SEEDS)
            + int(environment_seed)
        )
        ENVIRONMENT_SPECS.append(
            {
                "environment_index": len(ENVIRONMENT_SPECS),
                "source_environment_index": source_environment_index,
                "environment_id": (
                    f"structured__hegselmannkrause__"
                    f"{family}__env{environment_seed:02d}"
                ),
                "scenario_class": "structured_mechanism",
                "family": family,
                "dynamics": "hegselmannkrause",
                "environment_seed": int(environment_seed),
                "dynamics_params": {
                    "hk_epsilon": STRUCTURED_HK_EPSILON,
                    "hk_include_self": True,
                },
                "raw": raw,
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            }
        )

assert len(ENVIRONMENT_SPECS) == 15
print(
    pd.DataFrame(ENVIRONMENT_SPECS)[
        ["environment_index", "environment_id", "family", "environment_seed"]
    ].to_string(index=False)
)


 environment_index                                         environment_id            family  environment_seed
                 0       structured__hegselmannkrause__ring_bridge__env00       ring_bridge                 0
                 1       structured__hegselmannkrause__ring_bridge__env01       ring_bridge                 1
                 2       structured__hegselmannkrause__ring_bridge__env02       ring_bridge                 2
                 3       structured__hegselmannkrause__ring_bridge__env03       ring_bridge                 3
                 4       structured__hegselmannkrause__ring_bridge__env04       ring_bridge                 4
                 5     structured__hegselmannkrause__paired_bridge__env00     paired_bridge                 0
                 6     structured__hegselmannkrause__paired_bridge__env01     paired_bridge                 1
                 7     structured__hegselmannkrause__paired_bridge__env02     paired_bridge                 2
          

In [4]:

def make_env(
    spec: dict[str, Any],
    *,
    seed: int,
    initial_opinions: np.ndarray | None = None,
) -> NetworkGraph:
    x_init = np.asarray(
        spec["x0"] if initial_opinions is None else initial_opinions,
        dtype=float,
    ).copy()

    return NetworkGraph(
        connectivity_matrix=np.asarray(spec["A"], dtype=float).copy(),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=x_init,
        control_resistance=np.zeros(N, dtype=float),
        dynamics_model="hegselmannkrause",
        hk_epsilon=float(STRUCTURED_HK_EPSILON),
        hk_include_self=True,
        budget=1000.0,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )


def set_state(env: NetworkGraph, x0: np.ndarray) -> None:
    env.reset()
    env.opinions = np.asarray(x0, dtype=float).copy()
    if hasattr(env, "state"):
        try:
            env.state = np.asarray(x0, dtype=float).copy()
        except Exception:
            pass


def sample_structured_hk_x0(rng: np.random.Generator) -> np.ndarray:
    base = np.array(
        [
            0.35, 0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
            0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71,
        ],
        dtype=float,
    )
    return np.clip(
        base
        + rng.uniform(
            -HK_OPINION_JITTER,
            HK_OPINION_JITTER,
            size=N,
        ),
        0.0,
        1.0,
    )


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def _atomic_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def _save_npz_atomic(path: Path, **arrays: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(tmp, path)


def _save_torch_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def torch_load_compat(path: Path) -> Any:
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def model_from_state_file(path: Path) -> GraphIdentifierEnv:
    payload = torch_load_compat(path)
    kwargs = dict(payload.get("identifier_kwargs", IDENTIFIER_KWARGS))
    with contextlib.redirect_stdout(io.StringIO()):
        model = GraphIdentifierEnv(
            N=N,
            s=T_S,
            l2_lambda=0.0,
            zero_diag=True,
            **kwargs,
        )
    model.load_state_dict(payload["state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


def fit_model(
    X: np.ndarray,
    Y: np.ndarray,
    *,
    learning_seed: int,
    max_steps: int,
) -> tuple[GraphIdentifierEnv, np.ndarray, dict[str, Any]]:
    seed_everything(learning_seed)
    with contextlib.redirect_stdout(io.StringIO()):
        model = GraphIdentifierEnv(
            N=N,
            s=T_S,
            l2_lambda=0.0,
            zero_diag=True,
            **IDENTIFIER_KWARGS,
        )
    model.to(DEVICE)

    Xt = torch.tensor(
        np.asarray(X),
        dtype=torch.float32,
        device=DEVICE,
    )
    Yt = torch.tensor(
        np.asarray(Y),
        dtype=torch.float32,
        device=DEVICE,
    )
    n = int(Xt.shape[0])
    if n <= 0:
        raise ValueError("No training pairs.")

    opt = torch.optim.Adam(model.parameters(), lr=FIT_LR)
    t0 = time.perf_counter()
    stop_reason = "max_steps"
    steps_run = 0
    last_mae = np.nan

    for step in range(int(max_steps)):
        idx = torch.randint(
            0,
            n,
            (min(FIT_BATCH_SIZE, n),),
            device=DEVICE,
        )
        loss, _ = model.loss(Xt[idx], Yt[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        steps_run = step + 1

        if (
            step % FIT_CHECK_EVERY == 0
            or step == int(max_steps) - 1
        ):
            with torch.no_grad():
                last_mae = float(
                    (model.predict_next(Xt) - Yt)
                    .abs()
                    .mean()
                    .cpu()
                    .item()
                )
            if last_mae <= FIT_MAE_STOP:
                stop_reason = "mae_stop"
                break

    elapsed = time.perf_counter() - t0
    model.eval()
    with torch.no_grad():
        pred = model.predict_next(Xt)
        train_mae = float(
            (pred - Yt).abs().mean().cpu().item()
        )
        identity_mae = float(
            (Xt - Yt).abs().mean().cpu().item()
        )
        A_hat = model.A_hat().detach().cpu().numpy()

    return model, np.asarray(A_hat, dtype=float), {
        "fit_elapsed_s": float(elapsed),
        "fit_steps_run": int(steps_run),
        "fit_stop_reason": stop_reason,
        "train_mae": train_mae,
        "identity_mae": identity_mae,
        "model_over_identity": (
            float(train_mae / identity_mae)
            if identity_mae > 1e-12
            else np.nan
        ),
        "max_steps_requested": int(max_steps),
    }


def rollout_frozen_model(
    spec: dict[str, Any],
    model: GraphIdentifierEnv,
) -> dict[str, Any]:
    env = make_env(
        spec,
        seed=700_000 + int(spec["source_environment_index"]),
    )
    set_state(env, spec["x0"])
    state = np.asarray(spec["x0"], dtype=float).copy()
    states = [state.copy()]
    actions = []
    rewards = []
    inters = []
    max_u_vec = np.asarray(env.max_u, dtype=float).reshape(-1)

    for campaign in range(NUM_CAMPAIGNS):
        if campaign == 0:
            action = np.zeros(N, dtype=float)
        else:
            scores, _, _ = online_single_shot.learned_lambda_mix_scores(
                model,
                state,
                desired_opinion=TARGET,
                lambda_mix=LAMBDA_MIX,
                device=DEVICE,
            )
            action = online_single_shot.waterfill_from_scores(
                scores,
                max_u=max_u_vec,
                budget=float(B_CAMPAIGN),
            )

        x_next, reward, done, trunc, info = env.step(action)
        state = np.asarray(x_next, dtype=float).copy()
        states.append(state.copy())
        actions.append(np.asarray(action, dtype=float).copy())
        rewards.append(float(reward))
        inters.append(
            np.asarray(info["intermediate_states"], dtype=float).copy()
        )
        if done or trunc:
            break

    return {
        "states": np.asarray(states, dtype=float),
        "actions": np.asarray(actions, dtype=float),
        "rewards": np.asarray(rewards, dtype=float),
        "intermediate_states_list": inters,
    }


def rollout_summary(rollout: dict[str, Any]) -> dict[str, float]:
    states = np.asarray(rollout["states"], dtype=float)
    actions = np.asarray(rollout["actions"], dtype=float)
    controlled = actions[1:] if len(actions) > 1 else actions
    total = controlled.sum(axis=1) if len(controlled) else np.array([])
    hubs = controlled[:, [HK_LOW_HUB, HK_HIGH_HUB]].sum(axis=1) if len(controlled) else np.array([])
    with np.errstate(divide="ignore", invalid="ignore"):
        hub_frac = np.where(total > 1e-12, hubs / total, np.nan)

    return {
        "mean_end": float(states[-1].mean()),
        "min_end": float(states[-1].min()),
        "mean_over_campaigns": float(states.mean(axis=1).mean()),
        "mean_controlled_hub_budget_fraction": (
            float(np.nanmean(hub_frac))
            if len(hub_frac)
            else np.nan
        ),
    }


def pairs_from_rollout_inters(
    intermediate_states_list: list[np.ndarray],
) -> tuple[np.ndarray, np.ndarray]:
    xs = []
    ys = []
    for inter in intermediate_states_list:
        Xp, Yp = pairs_from_intermediate(
            np.asarray(inter, dtype=float)
        )
        xs.append(np.asarray(Xp, dtype=float))
        ys.append(np.asarray(Yp, dtype=float))
    return np.concatenate(xs, axis=0), np.concatenate(ys, axis=0)


In [5]:

# ---------- Recovery and HK-specific diagnostics ----------

def edge_average_precision(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    tol: float = 1e-12,
) -> float:
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    n = A_true.shape[0]
    mask = ~np.eye(n, dtype=bool)
    y = (A_true[mask] > tol).astype(int)
    scores = A_hat[mask]
    positives = int(y.sum())
    if positives == 0:
        return np.nan
    order = np.argsort(-scores, kind="mergesort")
    y_sorted = y[order]
    tp = np.cumsum(y_sorted)
    precision = tp / (np.arange(len(y_sorted)) + 1)
    return float(np.sum(precision * y_sorted) / positives)


def binary_average_precision(
    labels: np.ndarray,
    scores: np.ndarray,
) -> float:
    y = np.asarray(labels, dtype=int).reshape(-1)
    s = np.asarray(scores, dtype=float).reshape(-1)
    valid = np.isfinite(s)
    y = y[valid]
    s = s[valid]
    positives = int(y.sum())
    if positives == 0:
        return np.nan
    order = np.argsort(-s, kind="mergesort")
    ys = y[order]
    tp = np.cumsum(ys)
    precision = tp / (np.arange(len(ys)) + 1)
    return float(np.sum(precision * ys) / positives)


def true_support_mass(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    tol: float = 1e-12,
) -> float:
    vals = []
    for i in range(A_true.shape[0]):
        support = np.asarray(A_true[i] > tol).copy()
        support[i] = False
        vals.append(float(A_hat[i, support].sum()))
    return float(np.mean(vals))


def recovery_metrics(
    A_true: np.ndarray,
    A_hat: np.ndarray,
    v_true: np.ndarray,
) -> dict[str, float]:
    A_true = np.asarray(A_true, dtype=float)
    A_hat = np.asarray(A_hat, dtype=float)
    diff = A_hat - A_true
    v_hat = centrality_from_A(A_hat)
    row_tv = 0.5 * np.abs(diff).sum(axis=1)

    eps = 1e-12
    P = np.clip(A_hat, eps, None)
    row_entropy = -np.sum(P * np.log(P), axis=1)

    return {
        "A_MAE": float(np.mean(np.abs(diff))),
        "row_TV": float(np.mean(row_tv)),
        "true_edge_mass": true_support_mass(A_true, A_hat),
        "edge_AP": edge_average_precision(A_true, A_hat),
        "centrality_L1": float(
            np.sum(
                np.abs(
                    v_hat
                    - np.asarray(v_true, dtype=float)
                )
            )
        ),
        "effective_neighbors": float(
            np.mean(np.exp(row_entropy))
        ),
    }


def coverage_metrics(
    spec: dict[str, Any],
    X: np.ndarray,
) -> dict[str, float]:
    X = np.asarray(X, dtype=float)
    eps = float(spec["dynamics_params"]["hk_epsilon"])
    A = np.asarray(spec["A"], dtype=float)
    support = A > 1e-12
    np.fill_diagonal(support, False)

    d = np.abs(
        X[:, :, None]
        - X[:, None, :]
    )
    edge_d = d[:, support]

    bridge_gap = np.abs(
        X[:, HK_LOW_HUB]
        - X[:, HK_HIGH_HUB]
    )

    return {
        "n_states": int(len(X)),
        "edge_active_fraction": float(
            np.mean(edge_d <= eps)
        ),
        "edge_near_eps_0p025_fraction": float(
            np.mean(np.abs(edge_d - eps) <= 0.025)
        ),
        "edge_near_eps_0p05_fraction": float(
            np.mean(np.abs(edge_d - eps) <= 0.05)
        ),
        "edge_distance_mean": float(np.mean(edge_d)),
        "bridge_active_fraction": float(
            np.mean(bridge_gap <= eps)
        ),
        "bridge_near_eps_0p025_fraction": float(
            np.mean(np.abs(bridge_gap - eps) <= 0.025)
        ),
        "bridge_gap_mean": float(np.mean(bridge_gap)),
        "bridge_gap_q10": float(np.quantile(bridge_gap, 0.10)),
        "bridge_gap_q50": float(np.quantile(bridge_gap, 0.50)),
        "bridge_gap_q90": float(np.quantile(bridge_gap, 0.90)),
        "bridge_gap_min": float(np.min(bridge_gap)),
        "bridge_gap_max": float(np.max(bridge_gap)),
    }


def matrix_diagnostics(
    spec: dict[str, Any],
    model: GraphIdentifierEnv,
    X_ref: np.ndarray,
) -> dict[str, float]:
    X_ref = np.asarray(X_ref, dtype=float)
    A_true = np.asarray(spec["A"], dtype=float)
    eps = float(spec["dynamics_params"]["hk_epsilon"])

    model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(
            X_ref,
            dtype=torch.float32,
            device=DEVICE,
        )
        xi = Xt.unsqueeze(2)
        xj = Xt.unsqueeze(1)
        alpha = (
            model.alpha(xi, xj)
            .detach()
            .cpu()
            .numpy()
        )
        A_hat = (
            model.A_hat()
            .detach()
            .cpu()
            .numpy()
        )

    d = np.abs(
        X_ref[:, :, None]
        - X_ref[:, None, :]
    )
    gate = (d <= eps).astype(float)
    eye = np.eye(N, dtype=bool)
    gate[:, eye] = 0.0

    M_true = A_true[None, :, :] * gate
    M_full = A_hat[None, :, :] * alpha
    M_trueA_alpha = A_true[None, :, :] * alpha
    M_Ahat_gate = A_hat[None, :, :] * gate

    offdiag = ~eye
    physical = A_true > 1e-12
    np.fill_diagonal(physical, False)
    physical_3 = np.broadcast_to(
        physical[None, :, :],
        M_true.shape,
    )
    active = physical_3 & (gate > 0.5)
    inactive = physical_3 & (gate <= 0.5)
    nonedge = np.broadcast_to(
        ((~physical) & offdiag)[None, :, :],
        M_true.shape,
    )

    denom_mass = float(
        np.sum(M_full[:, offdiag])
    )
    if denom_mass <= 1e-12:
        denom_mass = np.nan

    labels = gate[:, physical]
    alpha_phys = alpha[:, physical]

    active_alpha = alpha[active]
    inactive_alpha = alpha[inactive]

    return {
        "M_MAE_all": float(
            np.mean(
                np.abs(
                    M_full[:, offdiag]
                    - M_true[:, offdiag]
                )
            )
        ),
        "M_MAE_true_support": float(
            np.mean(
                np.abs(
                    M_full[physical_3]
                    - M_true[physical_3]
                )
            )
        ),
        # Component-swap errors:
        # low trueA_alpha error => A_hat is the bottleneck;
        # low Ahat_gate error => alpha/gate learning is the bottleneck.
        "M_MAE_trueA_alpha": float(
            np.mean(
                np.abs(
                    M_trueA_alpha[:, offdiag]
                    - M_true[:, offdiag]
                )
            )
        ),
        "M_MAE_Ahat_true_gate": float(
            np.mean(
                np.abs(
                    M_Ahat_gate[:, offdiag]
                    - M_true[:, offdiag]
                )
            )
        ),
        "inactive_true_edge_mass_fraction": float(
            np.sum(M_full[inactive]) / denom_mass
        ),
        "spurious_nonedge_mass_fraction": float(
            np.sum(M_full[nonedge]) / denom_mass
        ),
        "alpha_gate_AP_on_true_edges": binary_average_precision(
            labels,
            alpha_phys,
        ),
        "alpha_active_mean": (
            float(np.mean(active_alpha))
            if active_alpha.size
            else np.nan
        ),
        "alpha_inactive_mean": (
            float(np.mean(inactive_alpha))
            if inactive_alpha.size
            else np.nan
        ),
        "alpha_active_over_inactive": (
            float(
                np.mean(active_alpha)
                / np.mean(inactive_alpha)
            )
            if (
                active_alpha.size
                and inactive_alpha.size
                and np.mean(inactive_alpha) > 1e-12
            )
            else np.nan
        ),
    }


In [6]:

# ---------- Reuse old abundant and single-shot artifacts ----------

_ABUNDANT_ENV_CACHE: dict[str, Path | None] = {}
_SINGLE_ENV_CACHE: dict[tuple[str, int], Path | None] = {}


def diagnostic_env_dir(spec: dict[str, Any]) -> Path:
    d = CACHE_ROOT / spec["environment_id"]
    d.mkdir(parents=True, exist_ok=True)
    return d


def find_abundant_source_env_dir(
    spec: dict[str, Any],
) -> Path | None:
    env_id = spec["environment_id"]
    if env_id in _ABUNDANT_ENV_CACHE:
        return _ABUNDANT_ENV_CACHE[env_id]

    hit = None
    for study in SOURCE_ABUNDANT_STUDIES:
        root = RESULTS_ROOT / "_trial_cache" / study
        if not root.exists():
            continue
        matches = [
            p for p in root.rglob(env_id)
            if p.is_dir()
        ]
        for candidate in matches:
            if (candidate / "dataset.npz").exists():
                hit = candidate
                break
        if hit is not None:
            break

    _ABUNDANT_ENV_CACHE[env_id] = hit
    return hit


def abundant_source_model_path(
    spec: dict[str, Any],
    seed: int,
) -> Path | None:
    d = find_abundant_source_env_dir(spec)
    if d is None:
        return None

    candidates = sorted(
        [
            *d.glob(
                f"learned_seed_{int(seed):02d}_model_state.pt"
            ),
            *d.glob(
                f"*seed_{int(seed):02d}*_model_state.pt"
            ),
        ]
    )
    return candidates[0] if candidates else None


def load_abundant_source_dataset(
    spec: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray] | None:
    d = find_abundant_source_env_dir(spec)
    if d is None:
        return None
    path = d / "dataset.npz"
    if not path.exists():
        return None
    with np.load(path) as arrays:
        return (
            np.asarray(arrays["X"], dtype=float),
            np.asarray(arrays["Y"], dtype=float),
        )


def find_single_shot_seed_dir(
    spec: dict[str, Any],
    seed: int,
) -> Path | None:
    key = (spec["environment_id"], int(seed))
    if key in _SINGLE_ENV_CACHE:
        return _SINGLE_ENV_CACHE[key]

    root = (
        RESULTS_ROOT
        / "_trial_cache"
        / SOURCE_SINGLE_SHOT_STUDY
    )
    hit = None
    if root.exists():
        env_dirs = [
            p
            for p in root.glob(
                f"{spec['environment_id']}__*"
            )
            if p.is_dir()
        ]
        for env_d in env_dirs:
            learned_root = env_d / "learned"
            if not learned_root.exists():
                continue
            seed_dirs = sorted(
                learned_root.glob(
                    f"seed_{int(seed):02d}_*"
                )
            )
            for candidate in seed_dirs:
                if (
                    (candidate / "rollout.npz").exists()
                    and (candidate / "summary.json").exists()
                ):
                    hit = candidate
                    break
            if hit is not None:
                break

    _SINGLE_ENV_CACHE[key] = hit
    return hit


def _load_single_npz(path: Path) -> dict[str, Any]:
    with np.load(path) as arrays:
        prefix = "learned_nonlinear"
        states = np.asarray(
            arrays[f"{prefix}__states"],
            dtype=float,
        )
        actions = np.asarray(
            arrays[f"{prefix}__actions"],
            dtype=float,
        )
        inter_keys = sorted(
            key
            for key in arrays.files
            if key.startswith(
                f"{prefix}__intermediate__"
            )
        )
        inters = [
            np.asarray(arrays[key], dtype=float)
            for key in inter_keys
        ]
    return {
        "states": states,
        "actions": actions,
        "intermediate_states_list": inters,
    }


def ensure_single_shot_reference(
    spec: dict[str, Any],
    seed: int,
) -> tuple[dict[str, Any], dict[str, Any], str]:
    local_d = diagnostic_env_dir(spec)
    local_npz = (
        local_d
        / f"single_shot_reference_seed_{int(seed):02d}.npz"
    )
    local_json = (
        local_d
        / f"single_shot_reference_seed_{int(seed):02d}.json"
    )

    if local_npz.exists() and local_json.exists():
        rollout = _load_single_npz(local_npz)
        summary = json.loads(
            local_json.read_text(encoding="utf-8")
        )
        return rollout, summary, "diagnostic_cache"

    source_d = find_single_shot_seed_dir(spec, seed)
    if source_d is not None:
        rollout = _load_single_npz(
            source_d / "rollout.npz"
        )
        summary = json.loads(
            (source_d / "summary.json").read_text(
                encoding="utf-8"
            )
        )
        return rollout, summary, "reused_final_single_shot"

    # Fallback: reproduce the exact final single-shot setup only for this HK case.
    train_seed = (
        700_000
        + 100 * int(spec["source_environment_index"])
        + int(seed)
    )
    rng_seed = (
        800_000
        + 100 * int(spec["source_environment_index"])
        + int(seed)
    )
    seed_everything(train_seed)
    env_template = make_env(
        spec,
        seed=900_000 + int(spec["source_environment_index"]),
    )
    result = online_single_shot.run_single_shot_online_identification(
        env_template,
        x0=np.asarray(spec["x0"], dtype=float),
        random_initial_opinions=False,
        num_campaigns_total=NUM_CAMPAIGNS,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        B_campaign=B_CAMPAIGN,
        lambda_mix=LAMBDA_MIX,
        exploration_campaigns=0,
        epsilon_schedule=[0.0] * NUM_CAMPAIGNS,
        lr=FIT_LR,
        l2_lambda=0.0,
        fit_max_steps=FIT_STEPS_STANDARD,
        fit_mae_stop=FIT_MAE_STOP,
        fit_batch_size=FIT_BATCH_SIZE,
        fit_check_every=FIT_CHECK_EVERY,
        identifier_kwargs=IDENTIFIER_KWARGS,
        device=DEVICE,
        rng_seed=rng_seed,
        suppress_fit_logs=True,
    )
    rollout = {
        "states": np.asarray(result["states"], dtype=float),
        "actions": np.asarray(result["actions"], dtype=float),
        "intermediate_states_list": [
            np.asarray(x, dtype=float)
            for x in result["intermediate_states_list"]
        ],
    }
    summary = {
        "environment_id": spec["environment_id"],
        "learning_seed": int(seed),
        **rollout_summary(rollout),
    }

    arrays = {
        "learned_nonlinear__states": rollout["states"],
        "learned_nonlinear__actions": rollout["actions"],
    }
    for campaign, inter in enumerate(
        rollout["intermediate_states_list"]
    ):
        arrays[
            f"learned_nonlinear__intermediate__{campaign:02d}"
        ] = inter
    _save_npz_atomic(local_npz, **arrays)
    _atomic_json(local_json, summary)
    return rollout, summary, "recomputed_single_shot_fallback"


In [7]:

# ---------- Data collection variants ----------

def collect_abundant_dataset(
    spec: dict[str, Any],
    *,
    policy: str,
    replay_actions: np.ndarray | None = None,
    replay_seed: int | None = None,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    # Keep x0 sampling deterministic within each collection policy.
    policy_offset = {
        "random": 0,
        "trueA": 1_000_000,
        "ss_replay": 2_000_000
        + 10_000 * int(0 if replay_seed is None else replay_seed),
    }[policy]
    base_seed = (
        DATA_SEED_BASE
        + 10_000 * int(spec["source_environment_index"])
        + policy_offset
    )
    rng = np.random.default_rng(base_seed)

    buf_x = []
    buf_y = []
    step_calls = 0
    t0 = time.perf_counter()

    for reset_idx in range(ABUNDANT_RESETS):
        x0_train = sample_structured_hk_x0(rng)
        env = make_env(
            spec,
            seed=base_seed + reset_idx,
            initial_opinions=x0_train,
        )
        set_state(env, x0_train)
        max_u_vec = np.asarray(
            env.max_u,
            dtype=float,
        ).reshape(-1)

        for campaign in range(ABUNDANT_CAMPAIGNS_PER_RESET):
            if campaign == 0:
                action = np.zeros(N, dtype=float)
            elif policy == "random":
                action = online_single_shot.waterfill_from_scores(
                    rng.random(N),
                    max_u=max_u_vec,
                    budget=float(B_CAMPAIGN),
                )
            elif policy == "trueA":
                action, _ = centrality_based_continuous_control(
                    env,
                    float(B_CAMPAIGN),
                    v=np.asarray(spec["v_true"], dtype=float),
                )
                action = np.asarray(action, dtype=float)
            elif policy == "ss_replay":
                if replay_actions is None:
                    raise ValueError(
                        "replay_actions required for ss_replay"
                    )
                action = np.asarray(
                    replay_actions[campaign],
                    dtype=float,
                )
            else:
                raise ValueError(policy)

            _, _, done, trunc, info = env.step(action)
            step_calls += 1
            inter = info.get("intermediate_states")
            if inter is None:
                raise RuntimeError(
                    "env.step did not return intermediate_states"
                )
            Xp, Yp = pairs_from_intermediate(
                np.asarray(inter, dtype=float)
            )
            buf_x.append(np.asarray(Xp, dtype=float))
            buf_y.append(np.asarray(Yp, dtype=float))
            if done or trunc:
                break

    X = np.concatenate(buf_x, axis=0)
    Y = np.concatenate(buf_y, axis=0)
    return X, Y, {
        "policy": policy,
        "n_pairs": int(len(X)),
        "n_resets": int(ABUNDANT_RESETS),
        "campaigns_per_reset": int(
            ABUNDANT_CAMPAIGNS_PER_RESET
        ),
        "env_step_calls": int(step_calls),
        "collection_elapsed_s": float(
            time.perf_counter() - t0
        ),
        "data_seed": int(base_seed),
    }


def dataset_cache_paths(
    spec: dict[str, Any],
    source: str,
    seed: int | None = None,
) -> tuple[Path, Path]:
    tag = source
    if seed is not None:
        tag += f"__seed_{int(seed):02d}"
    d = diagnostic_env_dir(spec)
    return (
        d / f"dataset__{tag}.npz",
        d / f"dataset__{tag}.json",
    )


def ensure_random_abundant_dataset(
    spec: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    local_npz, local_json = dataset_cache_paths(
        spec,
        "random_25x20",
    )
    if local_npz.exists():
        with np.load(local_npz) as arrays:
            X = np.asarray(arrays["X"], dtype=float)
            Y = np.asarray(arrays["Y"], dtype=float)
        meta = (
            json.loads(local_json.read_text(encoding="utf-8"))
            if local_json.exists()
            else {}
        )
        return X, Y, meta

    reused = load_abundant_source_dataset(spec)
    if reused is not None:
        X, Y = reused
        meta = {
            "policy": "random",
            "n_pairs": int(len(X)),
            "origin": "reused_abundant_25x20",
        }
        return X, Y, meta

    X, Y, meta = collect_abundant_dataset(
        spec,
        policy="random",
    )
    meta["origin"] = "collected_diagnostic_fallback"
    _save_npz_atomic(local_npz, X=X, Y=Y)
    _atomic_json(local_json, meta)
    return X, Y, meta


def ensure_trueA_dataset(
    spec: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    npz_path, json_path = dataset_cache_paths(
        spec,
        "trueA_25x20",
    )
    if npz_path.exists() and json_path.exists():
        with np.load(npz_path) as arrays:
            X = np.asarray(arrays["X"], dtype=float)
            Y = np.asarray(arrays["Y"], dtype=float)
        return X, Y, json.loads(
            json_path.read_text(encoding="utf-8")
        )

    X, Y, meta = collect_abundant_dataset(
        spec,
        policy="trueA",
    )
    _save_npz_atomic(npz_path, X=X, Y=Y)
    _atomic_json(json_path, meta)
    return X, Y, meta


def ensure_replay_dataset(
    spec: dict[str, Any],
    seed: int,
    actions: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    npz_path, json_path = dataset_cache_paths(
        spec,
        "ss_replay_25x20",
        seed,
    )
    if npz_path.exists() and json_path.exists():
        with np.load(npz_path) as arrays:
            X = np.asarray(arrays["X"], dtype=float)
            Y = np.asarray(arrays["Y"], dtype=float)
        return X, Y, json.loads(
            json_path.read_text(encoding="utf-8")
        )

    X, Y, meta = collect_abundant_dataset(
        spec,
        policy="ss_replay",
        replay_actions=np.asarray(actions, dtype=float),
        replay_seed=seed,
    )
    _save_npz_atomic(npz_path, X=X, Y=Y)
    _atomic_json(json_path, meta)
    return X, Y, meta


In [8]:

# ---------- Variant/model caching and evaluation ----------

def variant_paths(
    spec: dict[str, Any],
    seed: int,
    variant: str,
) -> tuple[Path, Path]:
    d = diagnostic_env_dir(spec)
    return (
        d / f"model__{variant}__seed_{int(seed):02d}.pt",
        d / f"row__{variant}__seed_{int(seed):02d}.json",
    )


def save_model_state(
    path: Path,
    model: GraphIdentifierEnv,
    *,
    seed: int,
    variant: str,
) -> None:
    _save_torch_atomic(
        path,
        {
            "state_dict": model.state_dict(),
            "identifier_kwargs": IDENTIFIER_KWARGS,
            "N": N,
            "s": T_S,
            "learning_seed": int(seed),
            "variant": variant,
        },
    )


def baseline_abundant_model(
    spec: dict[str, Any],
    seed: int,
    X_random: np.ndarray,
    Y_random: np.ndarray,
) -> tuple[GraphIdentifierEnv, np.ndarray, dict[str, Any], str]:
    source_path = abundant_source_model_path(spec, seed)
    if source_path is not None:
        model = model_from_state_file(source_path)
        with torch.no_grad():
            A_hat = (
                model.A_hat()
                .detach()
                .cpu()
                .numpy()
            )
            Xt = torch.tensor(
                X_random,
                dtype=torch.float32,
                device=DEVICE,
            )
            Yt = torch.tensor(
                Y_random,
                dtype=torch.float32,
                device=DEVICE,
            )
            train_mae = float(
                (model.predict_next(Xt) - Yt)
                .abs()
                .mean()
                .cpu()
                .item()
            )
            identity_mae = float(
                (Xt - Yt)
                .abs()
                .mean()
                .cpu()
                .item()
            )
        info = {
            "fit_elapsed_s": np.nan,
            "fit_steps_run": FIT_STEPS_STANDARD,
            "fit_stop_reason": "reused_source_model",
            "train_mae": train_mae,
            "identity_mae": identity_mae,
            "model_over_identity": (
                train_mae / identity_mae
                if identity_mae > 1e-12
                else np.nan
            ),
            "max_steps_requested": FIT_STEPS_STANDARD,
        }
        return (
            model,
            np.asarray(A_hat, dtype=float),
            info,
            "reused_abundant_25x20",
        )

    model, A_hat, info = fit_model(
        X_random,
        Y_random,
        learning_seed=seed,
        max_steps=FIT_STEPS_STANDARD,
    )
    return model, A_hat, info, "fitted_diagnostic_fallback"


def ensure_fitted_variant(
    spec: dict[str, Any],
    seed: int,
    variant: str,
    X: np.ndarray,
    Y: np.ndarray,
    *,
    max_steps: int,
) -> tuple[GraphIdentifierEnv, np.ndarray, dict[str, Any]]:
    model_path, row_path = variant_paths(
        spec,
        seed,
        variant,
    )
    if model_path.exists() and row_path.exists():
        model = model_from_state_file(model_path)
        with torch.no_grad():
            A_hat = (
                model.A_hat()
                .detach()
                .cpu()
                .numpy()
            )
        info = json.loads(
            row_path.read_text(encoding="utf-8")
        ).get("fit_info", {})
        return model, np.asarray(A_hat, dtype=float), info

    model, A_hat, info = fit_model(
        X,
        Y,
        learning_seed=seed,
        max_steps=max_steps,
    )
    save_model_state(
        model_path,
        model,
        seed=seed,
        variant=variant,
    )
    _atomic_json(row_path, {"fit_info": info})
    return model, A_hat, info


def evaluated_model_row(
    spec: dict[str, Any],
    seed: int,
    variant: str,
    model: GraphIdentifierEnv,
    A_hat: np.ndarray,
    fit_info: dict[str, Any],
    X_ref: np.ndarray,
    *,
    train_pairs: int,
    source_note: str,
) -> dict[str, Any]:
    rollout = rollout_frozen_model(spec, model)
    return {
        "environment_id": spec["environment_id"],
        "environment_index": int(spec["environment_index"]),
        "family": spec["family"],
        "environment_seed": int(spec["environment_seed"]),
        "learning_seed": int(seed),
        "variant": variant,
        "source_note": source_note,
        "train_pairs": int(train_pairs),
        **fit_info,
        **rollout_summary(rollout),
        **recovery_metrics(
            np.asarray(spec["A"], dtype=float),
            np.asarray(A_hat, dtype=float),
            np.asarray(spec["v_true"], dtype=float),
        ),
        **matrix_diagnostics(spec, model, X_ref),
    }


def single_shot_row(
    spec: dict[str, Any],
    seed: int,
    rollout: dict[str, Any],
    summary: dict[str, Any],
    source_note: str,
) -> dict[str, Any]:
    row = {
        "environment_id": spec["environment_id"],
        "environment_index": int(spec["environment_index"]),
        "family": spec["family"],
        "environment_seed": int(spec["environment_seed"]),
        "learning_seed": int(seed),
        "variant": "single_shot_online",
        "source_note": source_note,
        "train_pairs": np.nan,
        "fit_elapsed_s": np.nan,
        "fit_steps_run": np.nan,
        "fit_stop_reason": "online_repeated_fit",
        "train_mae": summary.get(
            "final_fit_train_mae",
            np.nan,
        ),
        "identity_mae": summary.get(
            "final_fit_identity_mae",
            np.nan,
        ),
        "model_over_identity": summary.get(
            "final_fit_model_over_identity",
            np.nan,
        ),
        "A_MAE": summary.get("A_MAE", np.nan),
        "row_TV": summary.get("row_TV", np.nan),
        "true_edge_mass": summary.get(
            "true_edge_mass",
            np.nan,
        ),
        "edge_AP": summary.get("edge_AP", np.nan),
        "centrality_L1": summary.get(
            "centrality_L1",
            np.nan,
        ),
        "effective_neighbors": summary.get(
            "effective_neighbors",
            np.nan,
        ),
    }
    row.update(rollout_summary(rollout))
    # No alpha model state was stored by the old single-shot cache.
    for key in [
        "M_MAE_all",
        "M_MAE_true_support",
        "M_MAE_trueA_alpha",
        "M_MAE_Ahat_true_gate",
        "inactive_true_edge_mass_fraction",
        "spurious_nonedge_mass_fraction",
        "alpha_gate_AP_on_true_edges",
        "alpha_active_mean",
        "alpha_inactive_mean",
        "alpha_active_over_inactive",
    ]:
        row[key] = np.nan
    return row


In [9]:

# ---------- Worker ----------

print("Structured HK environments:", len(ENVIRONMENT_SPECS))
print(
    "Potential new fits/environment/seed:",
    5,
    "(baseline 1k is reused when possible)",
)


def run_worker() -> None:
    assigned = [
        spec
        for spec in ENVIRONMENT_SPECS
        if int(spec["environment_index"]) % NUM_SHARDS == SHARD_ID
    ]
    print("Assigned environments:", len(assigned))

    all_rows = []
    coverage_rows = []
    t0 = time.perf_counter()

    for pos, spec in enumerate(assigned, start=1):
        env_t0 = time.perf_counter()
        print(
            f"\n[{pos}/{len(assigned)}] {spec['environment_id']}",
            flush=True,
        )

        # Current corrected abundant random dataset (reuse old 25x20 cache).
        X_random, Y_random, random_meta = (
            ensure_random_abundant_dataset(spec)
        )
        coverage_rows.append(
            {
                "environment_id": spec["environment_id"],
                "family": spec["family"],
                "environment_seed": int(spec["environment_seed"]),
                "learning_seed": np.nan,
                "data_source": "abundant_random_25x20",
                **coverage_metrics(spec, X_random),
            }
        )

        # True-A control collection: tests whether random-control state coverage is the issue.
        X_trueA, Y_trueA, trueA_meta = ensure_trueA_dataset(spec)
        coverage_rows.append(
            {
                "environment_id": spec["environment_id"],
                "family": spec["family"],
                "environment_seed": int(spec["environment_seed"]),
                "learning_seed": np.nan,
                "data_source": "abundant_trueA_control_25x20",
                **coverage_metrics(spec, X_trueA),
            }
        )

        for seed in LEARNING_SEEDS:
            ss_rollout, ss_summary, ss_origin = (
                ensure_single_shot_reference(spec, seed)
            )
            X_ss, Y_ss = pairs_from_rollout_inters(
                ss_rollout["intermediate_states_list"]
            )
            coverage_rows.append(
                {
                    "environment_id": spec["environment_id"],
                    "family": spec["family"],
                    "environment_seed": int(spec["environment_seed"]),
                    "learning_seed": int(seed),
                    "data_source": "single_shot_exact",
                    **coverage_metrics(spec, X_ss),
                }
            )

            X_replay, Y_replay, replay_meta = (
                ensure_replay_dataset(
                    spec,
                    seed,
                    ss_rollout["actions"],
                )
            )
            coverage_rows.append(
                {
                    "environment_id": spec["environment_id"],
                    "family": spec["family"],
                    "environment_seed": int(spec["environment_seed"]),
                    "learning_seed": int(seed),
                    "data_source": "abundant_single_shot_action_replay_25x20",
                    **coverage_metrics(spec, X_replay),
                }
            )

            # Common reference distribution for M/alpha diagnostics:
            # the exact successful single-shot trajectory.
            X_ref = X_ss

            # 1) Current abundant model, reused when available.
            model, A_hat, info, origin = baseline_abundant_model(
                spec,
                seed,
                X_random,
                Y_random,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "abundant_random_1k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_random),
                    source_note=origin,
                )
            )

            # 2) Same abundant data, much larger optimization budget.
            model, A_hat, info = ensure_fitted_variant(
                spec,
                seed,
                "abundant_random_20k",
                X_random,
                Y_random,
                max_steps=FIT_STEPS_LONG,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "abundant_random_20k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_random),
                    source_note="same_random_data_more_optimization",
                )
            )

            # 3) Same amount of abundant data, but collected under true-A control.
            model, A_hat, info = ensure_fitted_variant(
                spec,
                seed,
                "abundant_trueA_data_1k",
                X_trueA,
                Y_trueA,
                max_steps=FIT_STEPS_STANDARD,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "abundant_trueA_data_1k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_trueA),
                    source_note="25x20_trueA_control_collection",
                )
            )

            # 4) Same amount of abundant data, replaying the successful single-shot actions.
            model, A_hat, info = ensure_fitted_variant(
                spec,
                seed,
                "abundant_ss_replay_data_1k",
                X_replay,
                Y_replay,
                max_steps=FIT_STEPS_STANDARD,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "abundant_ss_replay_data_1k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_replay),
                    source_note="25x20_single_shot_action_replay",
                )
            )

            # 5) Fit once, offline, on exactly the data seen by single-shot.
            model, A_hat, info = ensure_fitted_variant(
                spec,
                seed,
                "ss_exact_offline_1k",
                X_ss,
                Y_ss,
                max_steps=FIT_STEPS_STANDARD,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "ss_exact_offline_1k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_ss),
                    source_note="exact_single_shot_data_one_fit",
                )
            )

            # 6) Same exact single-shot data, but approximate the cumulative
            # optimization budget of 20 online refits.
            model, A_hat, info = ensure_fitted_variant(
                spec,
                seed,
                "ss_exact_offline_20k",
                X_ss,
                Y_ss,
                max_steps=FIT_STEPS_LONG,
            )
            all_rows.append(
                evaluated_model_row(
                    spec,
                    seed,
                    "ss_exact_offline_20k",
                    model,
                    A_hat,
                    info,
                    X_ref,
                    train_pairs=len(X_ss),
                    source_note="exact_single_shot_data_long_fit",
                )
            )

            # 7) Original successful online single-shot result.
            all_rows.append(
                single_shot_row(
                    spec,
                    seed,
                    ss_rollout,
                    ss_summary,
                    ss_origin,
                )
            )

            print(
                f"  seed {seed}: "
                f"SS={all_rows[-1]['mean_end']:.4f} | "
                f"baseline={all_rows[-7]['mean_end']:.4f} | "
                f"random20k={all_rows[-6]['mean_end']:.4f} | "
                f"trueAdata={all_rows[-5]['mean_end']:.4f} | "
                f"replay={all_rows[-4]['mean_end']:.4f} | "
                f"ss1k={all_rows[-3]['mean_end']:.4f} | "
                f"ss20k={all_rows[-2]['mean_end']:.4f}",
                flush=True,
            )

        elapsed = time.perf_counter() - env_t0
        remaining = (
            (time.perf_counter() - t0) / pos
        ) * (len(assigned) - pos)
        print(
            f"  env elapsed {elapsed:.1f}s | "
            f"projected shard remaining {remaining / 60:.1f} min",
            flush=True,
        )

    pd.DataFrame(all_rows).to_csv(
        RESULTS_DIR / f"worker_shard_{SHARD_ID}_summary.csv",
        index=False,
    )
    pd.DataFrame(coverage_rows).to_csv(
        RESULTS_DIR / f"worker_shard_{SHARD_ID}_coverage.csv",
        index=False,
    )
    _atomic_json(
        RESULTS_DIR / f"SHARD_{SHARD_ID}_COMPLETE.json",
        {
            "status": "success",
            "shard_id": int(SHARD_ID),
            "n_environments": len(assigned),
            "n_result_rows": len(all_rows),
            "n_coverage_rows": len(coverage_rows),
            "elapsed_s": float(time.perf_counter() - t0),
            "git_commit": GIT_COMMIT,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        },
    )
    print(f"\nShard {SHARD_ID} complete.")


if WORKER_MODE:
    run_worker()
else:
    print(
        "Check mode only. Run the parallel launcher for computation."
    )


Structured HK environments: 15
Potential new fits/environment/seed: 5 (baseline 1k is reused when possible)
Check mode only. Run the parallel launcher for computation.


In [10]:

# ---------- Final merge / validation ----------

if WORKER_MODE:
    print("Worker mode: skipping final merge.")
else:
    shard_summaries = [
        RESULTS_DIR / f"worker_shard_{i}_summary.csv"
        for i in range(NUM_SHARDS)
    ]
    shard_coverages = [
        RESULTS_DIR / f"worker_shard_{i}_coverage.csv"
        for i in range(NUM_SHARDS)
    ]
    done_files = [
        RESULTS_DIR / f"SHARD_{i}_COMPLETE.json"
        for i in range(NUM_SHARDS)
    ]

    missing = [
        str(p)
        for p in (
            shard_summaries
            + shard_coverages
            + done_files
        )
        if not p.exists()
    ]
    print("Missing shard artifacts:", len(missing))

    if not missing:
        result_df = pd.concat(
            [pd.read_csv(p) for p in shard_summaries],
            ignore_index=True,
        )
        coverage_df = pd.concat(
            [pd.read_csv(p) for p in shard_coverages],
            ignore_index=True,
        )

        expected_result_rows = (
            len(ENVIRONMENT_SPECS)
            * len(LEARNING_SEEDS)
            * len(VARIANT_ORDER)
        )
        assert len(result_df) == expected_result_rows, (
            len(result_df),
            expected_result_rows,
        )

        # Coverage rows:
        # random + trueA once/environment = 30
        # exact SS + replay per seed = 90
        expected_coverage_rows = (
            len(ENVIRONMENT_SPECS) * 2
            + len(ENVIRONMENT_SPECS)
            * len(LEARNING_SEEDS)
            * 2
        )
        assert len(coverage_df) == expected_coverage_rows, (
            len(coverage_df),
            expected_coverage_rows,
        )

        result_df.to_csv(
            RESULTS_DIR / "combined_summary.csv",
            index=False,
        )
        coverage_df.to_csv(
            RESULTS_DIR / "coverage_summary.csv",
            index=False,
        )

        print("Result rows:", len(result_df))
        print("Coverage rows:", len(coverage_df))
        print(
            result_df.groupby("variant")["mean_end"]
            .agg(["mean", "std"])
            .round(6)
            .to_string()
        )
        print()
        print(
            coverage_df.groupby("data_source")[
                [
                    "bridge_active_fraction",
                    "bridge_near_eps_0p025_fraction",
                    "edge_active_fraction",
                ]
            ]
            .mean()
            .round(6)
            .to_string()
        )

        _atomic_json(
            RESULTS_DIR / "COMPUTATION_COMPLETE.json",
            {
                "status": "success",
                "study_name": STUDY_NAME,
                "pipeline_version": PIPELINE_VERSION,
                "git_commit": GIT_COMMIT,
                "n_environments": len(ENVIRONMENT_SPECS),
                "n_learning_seeds": len(LEARNING_SEEDS),
                "n_result_rows": len(result_df),
                "n_coverage_rows": len(coverage_df),
                "completed_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
        )
        print("Structured HK diagnostic complete.")
    else:
        print("First missing artifacts:", missing[:10])
        print("Run/re-run the parallel launcher. Cached fits are reused.")


Missing shard artifacts: 0
Result rows: 315
Coverage rows: 120
                                mean       std
variant                                       
abundant_random_1k          0.622440  0.003421
abundant_random_20k         0.895149  0.003847
abundant_ss_replay_data_1k  0.873950  0.025647
abundant_trueA_data_1k      0.888360  0.004389
single_shot_online          0.876433  0.005853
ss_exact_offline_1k         0.867118  0.027826
ss_exact_offline_20k        0.894547  0.004028

                                          bridge_active_fraction  bridge_near_eps_0p025_fraction  edge_active_fraction
data_source                                                                                                           
abundant_random_25x20                                   0.492400                        0.033093              0.982716
abundant_single_shot_action_replay_25x20                0.851111                        0.003316              0.994770
abundant_trueA_control_25x20         